CrossEncoder rerank as the final stage? This is precisely the failure mode CrossEncoders are good at correcting, because they attend over the query and candidate jointly rather than comparing two independently-pooled vectors. A CrossEncoder can learn that "HR professional" in the query needs to actually co-occur with HR-related terms in the candidate, not just generic seniority language — it's not limited to comparing two fixed points in embedding space. 

The first 'pass' is an SBERT model. SBERT embeddings for all candidates get computed once and cached as a single dense vector. It's a fast first pass that may pick up undesired candidates at first but is better than CrossEncode for first pass ranking whn catching semantics. CrossEncoder works well as a re-ranker once a first pass has happened.

In [1]:
import pandas as pd
from pathlib import Path

PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_PATH = RAW_DATA_DIR / "potential-talents - Aspiring human resources - seeking human resources.csv"
data = pd.read_csv(RAW_DATA_PATH)

In [2]:
# Data pre-processing:
data = data.drop_duplicates(subset=["job_title", "location", "connection", "fit"]).reset_index(drop=True) #All id's are unique, but the info in the other columns (when combined), aren't
data["id"] = data.index + 1
data["combined"] = (data["job_title"].astype(str)+ " located in " + data["location"].astype(str) + ", with " + data["connection"].astype(str)+ " connections.")

In [3]:
ideal_candidate_description = "Experienced in Human Resources"

In [8]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

def first_pass_retrieve(df, query, model, embeddings, top_k=25):
    """Cheap first pass: SBERT cosine similarity over cached candidate embeddings."""
    query_vec = model.encode([query])
    scores = cosine_similarity(query_vec, embeddings).flatten()

    result = df.copy()
    result["sbert_score"] = scores
    result = result.sort_values("sbert_score", ascending=False)
    return result.head(top_k).reset_index(drop=True)

def retrieve_then_rerank(df, query, sbert_model, embeddings, ce_model, first_pass_k=25, top_n=10):
    shortlist = first_pass_retrieve(df, query, sbert_model, embeddings, top_k=first_pass_k)

    pairs = [[query, desc] for desc in shortlist["job_title"].fillna("")]
    ce_scores = ce_model.predict(pairs, activation_fct=torch.nn.Sigmoid())

    shortlist["fit"] = ce_scores
    shortlist = shortlist.sort_values("fit", ascending=False)
    return shortlist.head(top_n).reset_index(drop=True)

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

candidate_texts = tuple(data["job_title"].fillna("").tolist())

def load_embeddings(_sbert_model, texts: tuple):
    return _sbert_model.encode(list(texts))

embeddings = load_embeddings(sbert_model, candidate_texts)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6562.58it/s]


In [9]:
two_step_rankings = retrieve_then_rerank(data, ideal_candidate_description,sbert_model,embeddings,ce_model,20,20)
two_step_rankings.head(20)

,id,job_title,location,connection,fit,combined,sbert_score
0,15,Experienced Retail Manager and aspiring Human ...,"Austin, Texas Area",57,0.997259,Experienced Retail Manager and aspiring Human ...,0.744926
1,25,Aspiring Human Resources Professional | Passio...,"New York, New York",212,0.747331,Aspiring Human Resources Professional | Passio...,0.694292
2,31,Aspiring Human Resources Professional | An ene...,"Austin, Texas Area",174,0.731421,Aspiring Human Resources Professional | An ene...,0.727404
3,22,"Aspiring Human Resources Manager, seeking inte...","Houston, Texas Area",7,0.525538,"Aspiring Human Resources Manager, seeking inte...",0.640563
4,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.312985,Aspiring Human Resources Specialist located in...,0.832370
5,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.190917,Aspiring Human Resources Professional located ...,0.839786
6,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.190917,Aspiring Human Resources Professional located ...,0.839786
7,21,Business Management Major and Aspiring Human R...,"Monroe, Louisiana Area",5,0.119485,Business Management Major and Aspiring Human R...,0.653065
8,16,"Human Resources, Staffing and Recruiting Profe...","Jackson, Mississippi Area",500+,0.083485,"Human Resources, Staffing and Recruiting Profe...",0.734504
9,13,Aspiring Human Resources Management student se...,"Houston, Texas Area",500+,0.067250,Aspiring Human Resources Management student se...,0.632509


Still not great at semantics. 

The best results so far are BERT model on the 'job_title' column rather than the 'combined' description.

Sentence BERT (SBERT) are 'all-MiniLM-L6-v2' and 'all-mpnet-base-v2' (tried in NB one). Let's try one again:

In [12]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')

def recommend_candidates_description_st(vectorizer,base_data, ideal_candidate_description, doc_column, top_n=10):
    """
    Recommends candidates based on similarity to an ideal candidate description.
    Only non-zero cosine similarities are returned.

    Use with SentenceTransformers
    """
    if "id" in base_data.columns:
        base_by_id = base_data.set_index("id", drop=False)
    else:
        base_by_id = base_data.copy()

    candidate_descriptions = base_by_id[doc_column].fillna("").tolist()
    candidate_vectors = vectorizer.encode(candidate_descriptions, convert_to_numpy=True)

    query_vector = vectorizer.encode([ideal_candidate_description])

    scores = cosine_similarity(query_vector, candidate_vectors).ravel()

    ranked = base_by_id.copy()
    ranked["fit"] = scores
    ranked = ranked[ranked["fit"] > 0].sort_values("fit", ascending=False)

    return ranked.head(top_n)

# doc column = combined
st_ideal_combined = recommend_candidates_description_st(st_model,data, ideal_candidate_description,"combined",20)
st_ideal_combined.head(10)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6866.18it/s]


,id,job_title,location,connection,fit,combined
id,,,,,,
25,25,Aspiring Human Resources Professional | Passio...,"New York, New York",212,0.681822,Aspiring Human Resources Professional | Passio...
31,31,Aspiring Human Resources Professional | An ene...,"Austin, Texas Area",174,0.666754,Aspiring Human Resources Professional | An ene...
6,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.651478,Aspiring Human Resources Specialist located in...
3,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.638040,Aspiring Human Resources Professional located ...
49,49,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,0.631948,Aspiring Human Resources Manager | Graduating ...
46,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.600609,Aspiring Human Resources Professional located ...
7,7,Student at Humber College and Aspiring Human R...,Kanada,61,0.589588,Student at Humber College and Aspiring Human R...
23,23,Human Resources Professional,Greater Boston Area,16,0.586130,Human Resources Professional located in Greate...
48,48,Seeking Human Resources Position,"Las Vegas, Nevada Area",48,0.573886,Seeking Human Resources Position located in La...


In [13]:
# doc column = job_title
st_ideal_job_title = recommend_candidates_description_st(st_model,data, ideal_candidate_description,"job_title",20)
st_ideal_job_title.head(10)

,id,job_title,location,connection,fit,combined
id,,,,,,
3,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.839786,Aspiring Human Resources Professional located ...
46,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.839786,Aspiring Human Resources Professional located ...
6,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.832370,Aspiring Human Resources Specialist located in...
23,23,Human Resources Professional,Greater Boston Area,16,0.831352,Human Resources Professional located in Greate...
15,15,Experienced Retail Manager and aspiring Human ...,"Austin, Texas Area",57,0.744926,Experienced Retail Manager and aspiring Human ...
48,48,Seeking Human Resources Position,"Las Vegas, Nevada Area",48,0.741169,Seeking Human Resources Position located in La...
16,16,"Human Resources, Staffing and Recruiting Profe...","Jackson, Mississippi Area",500+,0.734504,"Human Resources, Staffing and Recruiting Profe..."
31,31,Aspiring Human Resources Professional | An ene...,"Austin, Texas Area",174,0.727404,Aspiring Human Resources Professional | An ene...
37,37,Human Resources Management Major,"Milpitas, California",18,0.705285,Human Resources Management Major located in Mi...


Now running into the 'aspiring' and 'professional' semantic mix up with 'experienced'. Let's compare this with the TF-IDF method we I thought was the best in NB-1.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words="english")

In [17]:
def recommend_candidates_tf_idf(vectorizer,base_data, ideal_candidate_description,doc_column, top_n=10):
    """
    Recommends candidates based on similarity to an ideal candidate description.
    Only non-zero cosine similarities are returned.

    Use with TF-IDF model.
    """
    if "id" in base_data.columns:
        base_by_id = base_data.set_index("id", drop=False)
    else:
        base_by_id = base_data.copy()

    matrix = vectorizer.fit_transform(base_by_id[doc_column].fillna(""))
    ideal_vector = vectorizer.transform([ideal_candidate_description])

    scores = cosine_similarity(ideal_vector, matrix).ravel()

    ranked = base_by_id.copy()
    ranked["fit"] = scores
    ranked = ranked[ranked["fit"] > 0].sort_values("fit", ascending=False)

    return ranked.head(top_n)

# Doc column = "combined"
tf_idf_ideal_combined = recommend_candidates_tf_idf(tfidf_vectorizer, data, ideal_candidate_description, "combined", 20)
tf_idf_ideal_combined.head(10)

,id,job_title,location,connection,fit,combined
id,,,,,,
15,15,Experienced Retail Manager and aspiring Human ...,"Austin, Texas Area",57,0.475657,Experienced Retail Manager and aspiring Human ...
22,22,"Aspiring Human Resources Manager, seeking inte...","Houston, Texas Area",7,0.214073,"Aspiring Human Resources Manager, seeking inte..."
23,23,Human Resources Professional,Greater Boston Area,16,0.120795,Human Resources Professional located in Greate...
49,49,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,0.120654,Aspiring Human Resources Manager | Graduating ...
6,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.112862,Aspiring Human Resources Specialist located in...
46,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.110900,Aspiring Human Resources Professional located ...
14,14,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.107567,Seeking Human Resources Opportunities located ...
37,37,Human Resources Management Major,"Milpitas, California",18,0.107454,Human Resources Management Major located in Mi...
13,13,Aspiring Human Resources Management student se...,"Houston, Texas Area",500+,0.105763,Aspiring Human Resources Management student se...


In [18]:
# Doc column = "job_title"
tf_idf_ideal_job_title = recommend_candidates_tf_idf(tfidf_vectorizer, data, ideal_candidate_description, "job_title", 20)
tf_idf_ideal_job_title.head(10)

,id,job_title,location,connection,fit,combined
id,,,,,,
15,15,Experienced Retail Manager and aspiring Human ...,"Austin, Texas Area",57,0.621822,Experienced Retail Manager and aspiring Human ...
23,23,Human Resources Professional,Greater Boston Area,16,0.272441,Human Resources Professional located in Greate...
22,22,"Aspiring Human Resources Manager, seeking inte...","Houston, Texas Area",7,0.254132,"Aspiring Human Resources Manager, seeking inte..."
3,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.224467,Aspiring Human Resources Professional located ...
46,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.224467,Aspiring Human Resources Professional located ...
6,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.201210,Aspiring Human Resources Specialist located in...
14,14,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.189408,Seeking Human Resources Opportunities located ...
48,48,Seeking Human Resources Position,"Las Vegas, Nevada Area",48,0.189408,Seeking Human Resources Position located in La...
37,37,Human Resources Management Major,"Milpitas, California",18,0.176987,Human Resources Management Major located in Mi...


Honestly it's not as good as the BERT models carried out using the doc_column as 'job_title' rather than the 'combined' description.